# Qwen Text-to-SQL Setup (LLMSQL3 project)

Αυτό το notebook φορτώνει το **Qwen2.5-Coder-7B-Instruct** (4-bit quantized) και ορίζει μια function `generate_sql_qwen(question, schema_description)` με το ΙΔΙΟ interface όπως το `generate_sql_gpt()` στο τοπικό μας `llm_client.py`.

**Πριν τρέξεις:** Runtime → Change runtime type → **T4 GPU** (δωρεάν tier).

## 1. Εγκατάσταση dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

## 2. Φόρτωση μοντέλου (4-bit quantized, ώστε να χωράει στη δωρεάν T4 GPU)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import time
import re

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model (this can take a few minutes the first time)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
)
print("Model loaded.")

## 3. Το ίδιο system prompt / καθαρισμός εξόδου όπως στο llm_client.py (GPT-side)

Έτσι διασφαλίζουμε ότι τα δύο LLMs αξιολογούνται με **δίκαιο, ίδιο** prompt.

In [ ]:
SYSTEM_PROMPT = (
    "You are a text-to-SQL assistant. Given a database schema "
    "and a question in natural language, output ONLY the SQL query that answers "
    "the question. Do not include explanations, comments, or markdown formatting "
    "(no ```sql fences). Output a single valid SQL statement ending in a semicolon."
)

def _build_user_prompt(question, schema_description):
    return (
        f"Database schema:\n{schema_description}\n\n"
        f"Question: {question}\n\n"
        f"SQL query:"
    )

def _clean_sql_output(raw_text):
    text = raw_text.strip()
    text = re.sub(r"^```sql\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"```\s*$", "", text)
    return text.strip()

## 4. Function generate_sql_qwen -- ΙΔΙΟ interface με το generate_sql_gpt

In [ ]:
def generate_sql_qwen(question, schema_description, max_new_tokens=200):
    """
    Ίδιο interface με generate_sql_gpt() του llm_client.py:
    επιστρέφει dict με sql, latency_seconds, model, error.
    """
    user_prompt = _build_user_prompt(question, schema_description)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    start_time = time.time()
    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,  
                temperature=None,
                top_p=None,
                top_k=None,
            )
        elapsed = time.time() - start_time

        generated = output_ids[0][inputs["input_ids"].shape[1]:]
        raw_output = tokenizer.decode(generated, skip_special_tokens=True)
        sql = _clean_sql_output(raw_output)

        return {
            "sql": sql,
            "latency_seconds": elapsed,
            "model": "Qwen2.5-Coder-7B-Instruct-4bit",
            "error": None,
        }
    except Exception as e:
        elapsed = time.time() - start_time
        return {
            "sql": None,
            "latency_seconds": elapsed,
            "model": "Qwen2.5-Coder-7B-Instruct-4bit",
            "error": str(e),
        }

## 5. Γρήγορο test -- ΙΔΙΑ ερώτηση με το GPT test, για δίκαιη σύγκριση

In [ ]:
test_schema = """
Table: state(state_name, population, area, capital, density)
Table: city(city_name, population, state_name)
Table: river(river_name, length, traverse)
""".strip()

test_question = "What is the population of Texas?"

print(f"Question: {test_question}")
print("Calling Qwen...")
result = generate_sql_qwen(test_question, test_schema)

print()
print(f"Model: {result['model']}")
print(f"Latency: {result['latency_seconds']:.2f}s")
print(f"Error: {result['error']}")
print(f"SQL: {result['sql']}")

## 6. Batch mode: τρέξε το Qwen πάνω σε ΟΛΟ το δείγμα ερωτήσεων

Ανέβασε εδώ το `sample_for_qwen.csv` (παράχθηκε τοπικά από το `export_sample_for_qwen.py`) -- περιέχει τις ΙΔΙΕΣ ερωτήσεις που χρησιμοποιήσαμε ήδη για το GPT run, μαζί με το schema description της κάθε βάσης (ώστε το Colab να μη χρειάζεται καθόλου πρόσβαση στην τοπική μας MySQL/MariaDB).

In [ ]:
from google.colab import files

print("Επίλεξε το sample_for_qwen.csv από τον υπολογιστή σου:")
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
print(f"Uploaded: {csv_filename}")

In [ ]:
import pandas as pd

sample_df = pd.read_csv(csv_filename)
print(f"Loaded {len(sample_df)} rows")
print(sample_df['dataset'].value_counts())

In [ ]:
results = []

for i, row in sample_df.iterrows():
    result = generate_sql_qwen(row['question'], row['schema_description'])
    results.append({
        'dataset': row['dataset'],
        'difficulty': row['difficulty'],
        'question': row['question'],
        'gold_sql': row['gold_sql'],
        'generated_sql': result['sql'],
        'llm_model': result['model'],
        'generation_latency_seconds': result['latency_seconds'],
        'generation_error': result['error'],
    })
    if (i + 1) % 10 == 0:
        print(f"  [{i + 1}/{len(sample_df)}] processed")

results_df = pd.DataFrame(results)
print()
print("Done. Sample of results:")
print(results_df[['dataset', 'generated_sql']].head(10).to_string())

## 7. Αποθήκευση + κατέβασμα του αρχείου αποτελεσμάτων

Κατέβασε το `qwen_results.csv` και τρέξε το τοπικά με το `src/score_qwen_results.py` (θα εκτελέσει το SQL στη δική σου MySQL/MariaDB και θα υπολογίσει accuracy, ίδιο μοτίβο με το GPT run).

In [ ]:
results_df.to_csv('qwen_results.csv', index=False)
files.download('qwen_results.csv')